# ConfRover — mini train → generate → eval (PACE)

A full end-to-end pass on the **4 bundled ATLAS proteins**, offline. This is a
plumbing test, not a real model: a few hundred steps on 4 proteins won't learn
much, but it exercises the exact code paths you'll scale up (see
`src/confrover/train/RUNBOOK.md` for the real ~50–100 protein run).

Steps: (1) build a training dataset, (2) train a few hundred steps with a
visible loss curve, (3) confirm the checkpoint loads via the inference path,
(4) generate trajectories offline, (5) score them with `confrover eval`.

In [ ]:
import os, sys
from pathlib import Path

# Resolve repo root whether the notebook runs from examples/ or repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo root:", REPO_ROOT)

# Bundled test data -- everything below runs offline (no ATLAS download / MSA).
TESTDATA   = REPO_ROOT / "tests" / "test_data"
ATLAS_ROOT = TESTDATA / "atlas"            # <case>/<case>.pdb + <case>_prod_R1_fit.xtc
REPR_ROOT  = TESTDATA / "openfold_repr"    # cached OpenFold features (seqres-keyed)

# The four bundled proteins that have BOTH an ATLAS xtc and cached OpenFold repr.
BUNDLED = {
    "7jfl_C": "SALQDLLRTLKSPSSPQQQQQVLNILKSNPQLMAAFIKQRTAKYVAN",
    "6okd_C": "GSREGCASRCMKYNDELEKCEARMMSMSNTEEDCEQELEDLLYCLDHCHSQ",
    "6ro6_A": "MEGALPKGLSDLIADPTLGPQITPDWVRTLSRIELRGKRPRDKQDWYEIYLHLKRILS",
    "7lp1_A": "VTQSFLPPGWEMRIAPNGRPFFIDHNTKTTTWEDPRLKF",
}

CACHE = Path(os.environ.get("CONFROVER_CACHE_DIR", Path.home() / "scratch" / "confrover_cache")).expanduser()
CACHE.mkdir(parents=True, exist_ok=True)

# generate() unconditionally runs install_cutlass(), which git-clones CUTLASS if
# it's missing -- that fails on an offline compute node. Evo-attention kernels are
# OFF in these configs, so CUTLASS is never actually used; point CUTLASS_PATH at an
# existing (stub) dir so the clone is skipped.
os.environ.setdefault("CUTLASS_PATH", str(CACHE / "cutlass_stub"))
Path(os.environ["CUTLASS_PATH"]).mkdir(parents=True, exist_ok=True)

import torch
torch.set_float32_matmul_precision("high")
assert torch.cuda.is_available(), "No GPU visible -- request an interactive GPU (salloc) first."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Training dataset (4 bundled proteins)
Small strides so windows fit the short bundled trajectories; `samples_per_epoch` gives several windows per epoch.

In [ ]:
from confrover.train.dataset import _count_xtc_frames, TrajCaseConfig, TrajDatasetConfig, TrajDataset
from confrover.data.pretrain_repr.openfold.loader import OpenFoldReprLoader

nframes_avail = min(_count_xtc_frames(str(ATLAS_ROOT / c / f"{c}_prod_R1_fit.xtc")) for c in BUNDLED)
print("min trajectory length across bundled proteins:", nframes_avail)

N_FRAMES = 4
max_stride = max(1, (nframes_avail - 1) // (N_FRAMES - 1))
strides = sorted({s for s in [10, 20, 40] if s <= max_stride}) or [max_stride]
print("using strides:", strides)

cases = [
    TrajCaseConfig(case_id=c, seqres=s, pdb_fpath=f"{c}/{c}.pdb",
                   xtc_fpaths=[f"{c}/{c}_prod_R1_fit.xtc"])
    for c, s in BUNDLED.items()
]
train_cfg = TrajDatasetConfig(name="bundled4_train", n_frames=N_FRAMES,
                              stride_in_10ps=strides[0], strides_in_10ps=strides,
                              samples_per_epoch=64, cases=cases)
repr_loader = OpenFoldReprLoader(repr_root=str(REPR_ROOT), num_recycles=3,
                                 load_single=True, load_pair=True, v1=False)
train_ds = TrajDataset(config=train_cfg, repr_loader=repr_loader,
                       relpath_to=str(ATLAS_ROOT), deterministic=False,
                       batch_size=1, num_workers=0, shuffle=True)
loader = torch.utils.data.DataLoader(train_ds, batch_size=1, collate_fn=TrajDataset.collate)
print("windows/epoch:", len(train_ds))

## 2. Train (short manual loop with a loss curve)
A hand-rolled loop makes the loss easy to plot. The Lightning `Trainer` path (used at scale) is `python -m confrover.train.cli`; see the RUNBOOK.

In [ ]:
import hydra, itertools
from omegaconf import OmegaConf

model_cfg = OmegaConf.load(REPO_ROOT / "src" / "confrover" / "configs" / "model" / "confrover_train.yaml")
model_cfg.optimizer_cfg.lr = 3e-4
model = hydra.utils.instantiate(model_cfg).to("cuda")
model.set_model_cfg(OmegaConf.to_container(model_cfg, resolve=True))
model.train()

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=model.optimizer_cfg["lr"])
from lightning.pytorch.utilities import move_data_to_device

N_STEPS = 300
hist = []
data_iter = itertools.cycle(loader)
for step in range(N_STEPS):
    batch = move_data_to_device(next(data_iter), "cuda")
    opt.zero_grad(set_to_none=True)
    loss, aux = model._shared_step(batch, stage="train")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    hist.append(float(loss))
    if step % 25 == 0 or step == N_STEPS - 1:
        print(f"step {step:4d}  loss={hist[-1]:.4f}  rot={float(aux['loss_rot']):.3f}  "
              f"trans={float(aux['loss_trans']):.3f}  bb={float(aux.get('loss_bb_atom', 0.0)):.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist); ax[0].set_title("train loss"); ax[0].set_xlabel("step")
# smoothed view (loss is noisy because t is sampled per step)
k = 15
sm = np.convolve(hist, np.ones(k)/k, mode="valid")
ax[1].plot(sm); ax[1].set_title(f"train loss ({k}-step moving avg)"); ax[1].set_xlabel("step")
plt.tight_layout(); plt.show()
print(f"first={hist[0]:.3f}  last(avg)={sm[-1]:.3f}")
print("NB: 4 proteins x 300 steps is a plumbing test -- expect only a mild, noisy decrease.")

## 3. Save a checkpoint and confirm it loads via the inference path

In [ ]:
from confrover.model.confrover import ConfRover
from confrover.utils.torch.ckpt import load_model_checkpoint

ckpt = {"state_dict": model.state_dict()}
model.on_save_checkpoint(ckpt)                 # embeds inference-shaped model_cfg
ckpt_path = CACHE / "bundled4_trained.pt"
torch.save(ckpt, ckpt_path)

# The upstream loader: build a plain ConfRover from the embedded cfg, load weights.
inf_model = ConfRover.from_config(ckpt["model_cfg"]).to("cuda")
inf_model = load_model_checkpoint(inf_model, ckpt_path, strict=True)
print("OK: trained checkpoint loads into a plain ConfRover via load_model_checkpoint")
print("ckpt:", ckpt_path)

## 4. Generate trajectories (offline)
Uses the cached bundled OpenFold repr, so `generate_repr` skips the MSA server, and the CUTLASS clone is skipped by the `CUTLASS_PATH` stub set in the setup cell. Forward simulation from each protein's frame-0 PDB.

> If a cell here reaches for the network, you're on a node without internet and something wasn't cached — pre-run the OpenFold-feature step on a login node first (see the RUNBOOK).

In [ ]:
from confrover.model.decoder.confdiff.sampler.euler import EulerSampler

inf_model.decoder.sampler = EulerSampler(diffusion_steps=20)
inf_model.eval()

GEN_DIR = CACHE / "bundled4_gen"
for cid, seq in BUNDLED.items():
    inf_model.generate(
        case_id=cid, seqres=seq, task_mode="forward",
        output_dir=str(GEN_DIR),
        conditions=str(ATLAS_ROOT / cid / f"{cid}.pdb"),
        n_frames=20, stride_in_10ps=strides[0], n_replicates=3,
        cache_dir=str(CACHE), folding_repr=str(REPR_ROOT), diffusion_steps=20,
    )
print("Generated under:", GEN_DIR)
print(sorted(p.name for p in GEN_DIR.iterdir()))

## 5. Evaluate: ATLAS metrics vs. the reference MD
Compares the generated ensemble to the bundled reference trajectory. **Numbers will be poor** — the model saw only a few hundred steps — but this proves the full generate→eval loop works. A trained model should push `rmsf_pearson` up and `coverage_mean_min_rmsd` down.

In [ ]:
from confrover.train.eval.metrics import evaluate_dir
import pandas as pd

rows = evaluate_dir(gen_dir=GEN_DIR, ref_dir=ATLAS_ROOT, output_dir=GEN_DIR,
                    contact_cutoff=8.0, max_pairwise=100)
df = pd.DataFrame(rows)
cols = ["case_id", "seqlen", "n_gen_frames", "n_ref_frames", "rmsf_pearson",
        "rmsf_mae", "rg_wasserstein", "contact_map_mae",
        "coverage_mean_min_rmsd", "coverage_mean_best_tmscore"]
display(df[[c for c in cols if c in df.columns]].round(3))
print("Wrote:", GEN_DIR / "metrics.csv")

✅ End-to-end pipeline exercised: dataset → training → `from_pretrained`-compatible
checkpoint → offline generation → quantitative metrics.

**To scale to a real run** (per `src/confrover/train/RUNBOOK.md`):
1. `python scripts/build_manifests.py` on a ~50–100 protein ATLAS subset (with all 3 replicates).
2. `python -m confrover.train.cli --train_manifest ... --val_manifest ... --output_dir ...` (or `scripts/phoenix_train.sbatch`).
3. `confrover generate --model <ckpt.pt> --job_config <eval_manifest> --output ...`
4. `confrover eval --gen_dir ... --ref_dir <atlas_root>`